In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from pathlib import Path
from functools import partial
from typing import Tuple, Callable

import numpy as onp
from natsort import natsorted
import numpy.typing as npt
import jax
import jax.scipy.optimize
import jax.numpy as jnp
from ase.atoms import Atoms
from ase.visualize import view
from matscipy.neighbours import neighbour_list

from clinamen2.utils.structure_setup import place_atoms_packmol
from msmjax.core.shortrange import make_eval_pair_pot

# Definitions

In [2]:
DENSITY = 1.0
N_DIM = 3
PACKMOL_TOLERANCE = 2 / 3

OUTDIR_STRUCTURES = Path("structures/")

In [3]:
def wrap_positions(positions, cell, pbcs):
    scaled = positions @ jnp.linalg.pinv(cell)
    wrapped_scaled = jnp.where(pbcs, scaled - scaled // 1.0, scaled)
    return wrapped_scaled @ cell


def shortrange_quadratic_potential(r, r_cut=1.0, prefactor=1.0):
    """Evaluate a short-range quadratic potential for scalar distance arguments

    Args:
        r: Scalar distance value
        r_cut: Cutoff radius of the potential
        prefactor: Prefactor of the potential

    Returns:
        Potential evaluated at distance r
    """
    return jnp.where(
        jnp.abs(r) < r_cut, prefactor * (jnp.abs(r) - r_cut) ** 2, 0.0
    )


def make_overlap_penalty_fn_pairs(
    box_lengths, pbc, min_dist=1.0, prefactor=1.0
) -> Callable:
    """Create a function that penalizes particles closer together than min_dist

    Args:
        box_lengths: Sequence of box lengths, one value per dimension
        pbc:
        min_dist: Threshold distance below which particle pair incur an energy
            penalty
        prefactor: Prefactor of the penalty potential

    Returns:
        A function that takes a set of vector-valued particle positions and
        computes the total energy penalty over all particle pairs
    """
    pbc = onp.asarray(pbc)
    cell = onp.diag(box_lengths)

    eval_pair_pot = make_eval_pair_pot(
        kernel_fn=partial(
            shortrange_quadratic_potential, r_cut=min_dist, prefactor=prefactor
        ),
        pbc=pbc,
        cell_mode="ortho",
    )

    def energy_fn(positions):
        n_particles = positions.shape[0]
        return eval_pair_pot(
            positions, charges=onp.ones(n_particles), cell=cell
        )

    return energy_fn


def make_minimization_target(energy_fn: Callable, n_dim: int) -> Callable:
    """Make a loss function suitable for scipy optimizers from an energy function

    Args:
        energy_fn: A function that takes in particle positions and returns the
            total system energy.

    Returns:
        A function of two arguments (each argument is one flat array of
        positions) that returns the total energy of the system. The point of
        the two different sets of positions is that the first will be considered
        as degrees of freedom by optimizers, and the second represents any fixed
        particles that might be present.
    """

    def target_fn(positions_dof_flat, positions_fixed_flat):
        positions_dof = positions_dof_flat.reshape((-1, n_dim))
        positions_fixed = positions_fixed_flat.reshape((-1, n_dim))
        positions_combined = jnp.concatenate([positions_dof, positions_fixed])
        return energy_fn(positions_combined)

    return target_fn


def make_optimize_positions(cell, min_dist):
    n_dim = len(cell)

    energy_fn = make_overlap_penalty_fn_pairs(
        box_lengths=onp.diag(cell),
        pbc=onp.array([True] * n_dim),
        min_dist=min_dist,
    )
    target_fn = make_minimization_target(energy_fn=energy_fn, n_dim=n_dim)

    @jax.jit
    def optimize_positions(positions_dof, positions_fixed):
        result = jax.scipy.optimize.minimize(
            fun=target_fn,
            x0=positions_dof.ravel(),
            args=(positions_fixed.ravel(),),
            method="BFGS",
        )
        return result.x.reshape((-1, n_dim)), result

    return optimize_positions

In [4]:
def postprocess_packmol_structure_for_pbcs(
    positions: npt.ArrayLike,
    cell: npt.ArrayLike,
    min_dist: float,
    optimize_positions_fn: Callable,
) -> Tuple[onp.ndarray, jax.scipy.optimize.OptimizeResults]:
    """Optimize particles near boundaries to avoid overlaps due to periodicity

    This is done by optimizing the positions of the atoms in an "outer shell"
    near the cell boundaries, while keeping atoms in an "inner shell" fixed, and
    ignoring all atoms in the center of the cell (to reduce computational cost).

    Args:
        positions: Array of positions of all particles in the system.
        cell: Unit cell
        min_dist: Minimum distance that particle pairs should have
        optimize_positions_fn: A function that returns optimized outer-shell
            positions, given original outer-shell positions, and inner-shell
            positions as arguments.

    Returns:
        2-element tuple containing:
            - Array of particle positions optimized to remove/reduce overlaps
            - `OptimizeResults` object containing more information about the
                optimization
    """
    approx_thickness_outer_shell = 1.25 * min_dist
    approx_thickness_inner_shell = 1.5 * min_dist

    volume = onp.linalg.det(cell)
    dnsty = positions.shape[0] / volume
    sidelength = cell[0, 0]
    n_particles_total = positions.shape[0]
    n_dim = positions.shape[1]

    wall_distances = onp.concatenate(
        [onp.abs(positions), onp.abs(onp.diag(cell) - positions)], axis=1
    )
    wall_distances = onp.min(wall_distances, axis=1)
    inds_sorted_by_wall_distance = onp.argsort(wall_distances)

    sidelength_center = sidelength - 2 * (
        approx_thickness_outer_shell + approx_thickness_inner_shell
    )
    sidelength_center_plus_inner = (
        sidelength - 2 * approx_thickness_outer_shell
    )
    volume_center = sidelength_center**3
    volume_center_plus_inner = sidelength_center_plus_inner**3
    n_particles_outer = min(
        int(dnsty * (volume - volume_center_plus_inner)) + 1, n_particles_total
    )
    n_particles_outer_plus_inner = min(
        int(dnsty * (volume - volume_center)) + 1, n_particles_total
    )
    inds_outer = inds_sorted_by_wall_distance[:n_particles_outer]
    inds_inner = inds_sorted_by_wall_distance[
        n_particles_outer:n_particles_outer_plus_inner
    ]

    pos_opt_outer_shell, result = optimize_positions_fn(
        positions[inds_outer], positions[inds_inner]
    )

    pos_opt = positions.copy()
    pos_opt[inds_outer] = pos_opt_outer_shell
    pos_opt = wrap_positions(
        pos_opt, cell=cell, pbcs=onp.array([True] * n_dim)
    )

    return onp.asarray(pos_opt), result

# Generate structures

In [5]:
OUTDIR_STRUCTURES.mkdir(parents=True)

for n_particles in onp.concatenate(
    [
        [50, 100, 200, 300, 400],
        onp.arange(500, 5001, 500),
        onp.arange(5000, 16000, 1000),
    ]
):
    sidelength = (n_particles / DENSITY) ** (1 / N_DIM)
    box_lengths = jnp.full(N_DIM, sidelength)
    cell = onp.diag(box_lengths)

    optimize_positions = make_optimize_positions(
        cell=cell, min_dist=PACKMOL_TOLERANCE
    )

    positions = []
    charges = []

    if n_particles in [100, 10000]:
        # Generate more structure for those numbers of particles
        # at which an accuracy benchmark is run
        n_structures = 11
    else:
        n_structures = 1

    for seed in range(n_structures):
        pos = place_atoms_packmol(
            n_atoms=n_particles,
            side_length=sidelength,
            tolerance=PACKMOL_TOLERANCE,
            exec_string="/home/florian/Downloads/packmol-20.14.2/packmol",
            random_seed=seed,
        )
        pos, _ = postprocess_packmol_structure_for_pbcs(
            positions=pos,
            cell=cell,
            min_dist=PACKMOL_TOLERANCE,
            optimize_positions_fn=optimize_positions,
        )
        positions.append(pos)

        rng = onp.random.default_rng(seed)
        chg = rng.uniform(low=-1.0, high=1.0, size=n_particles)
        chg -= chg.sum() / n_particles
        charges.append(chg)

    outdict = {
        "positions": onp.asarray(positions),
        "charges": onp.asarray(charges),
        "cells": onp.stack([cell] * len(positions)),
    }

    onp.savez(OUTDIR_STRUCTURES / f"structures_{n_particles}", **outdict)

tolerance 0.6666666666666666
filetype xyz 
output box_50.xyz
seed 0

structure trivial.xyz
  number 50
  inside box 0. 0. 0. 3.6840314986403864 3.6840314986403864 3.6840314986403864
end structure


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                              Version 20.14.2 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: xyz
  Seed for random number generator:            0
  Output file: box_50.xyz
  Reading coordinate file: trivial.xyz
  Number of independent structures:            1
  The structures are: 
  Structure            1 :sphere(           1  atoms)

# Make checks on the generated structures

## Distances

Check if the minimum distance is below the target in any of the structures, and if so, what is the actual minimum distance (it is okay for the distance to be below the target, if it's not by much; this can sometimes happen if the minimization doesn't find a perfect solution).

In [21]:
pbc = onp.array([True] * N_DIM)

for npzfile in natsorted(OUTDIR_STRUCTURES.glob("structures_*.npz")):
    print(npzfile)
    loaded = onp.load(npzfile)

    # All structures of the same particle number were created in the same cell
    cell = loaded["cells"][0]

    for idx_structure in range(len(loaded["positions"])):
        pos = loaded["positions"][idx_structure]
        chg = loaded["charges"][idx_structure]
        # A fast and convenient way of checking if any two particles are closer
        # together than the target distance is by constructing the neighbor list
        # for the structure, with the cutoff set to the target distance.
        i, j, d = neighbour_list(
            "ijd",
            cutoff=PACKMOL_TOLERANCE,
            positions=pos,
            cell=cell,
            pbc=(True,) * 3,
        )
        if len(d) > 0:
            min_dist = d.min()
            print(
                f"- Minimum distance below target: "
                f"target = {PACKMOL_TOLERANCE:.3f}, actual = {min_dist:.3f}"
            )

    print()

structures/structures_50.npz

structures/structures_100.npz
- Minimum distance below target: target = 0.667, actual = 0.662
- Minimum distance below target: target = 0.667, actual = 0.661

structures/structures_200.npz

structures/structures_300.npz

structures/structures_400.npz

structures/structures_500.npz

structures/structures_1000.npz

structures/structures_1500.npz

structures/structures_2000.npz

structures/structures_2500.npz

structures/structures_3000.npz

structures/structures_3500.npz

structures/structures_4000.npz

structures/structures_4500.npz

structures/structures_5000.npz

structures/structures_6000.npz

structures/structures_7000.npz

structures/structures_8000.npz

structures/structures_9000.npz

structures/structures_10000.npz

structures/structures_11000.npz

structures/structures_12000.npz

structures/structures_13000.npz

structures/structures_14000.npz

structures/structures_15000.npz



## Visual inspection

In [20]:
n_particles = 500
loaded = onp.load(OUTDIR_STRUCTURES / f"structures_{n_particles}.npz")
idx_structure = 0
cell = loaded["cells"][idx_structure]
pos = loaded["positions"][idx_structure]
atoms = Atoms(
    cell=cell,
    positions=pos,
    pbc=(True,) * N_DIM,
)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

Error processing line 1 of /home/florian/anaconda3/envs/msmjax_py3.11/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 195, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
